# Gate 0B - Source Reachability And First Acquisition

This notebook begins Evidence Gate 0B for the Phase 1 AI layer.

The goal is to prove which Priority 1 public sources are reachable, record the result in project ledgers, and protect the project from premature funding claims.

This notebook does **not** yet score routes, recommend tactical corridor segments, or select the pavilion site.

## Safety Rules

- Check official source pages/API endpoints first.
- Do not download national bulk datasets in this notebook.
- Do not mark any source as funding-ready from reachability alone.
- Record every raw file in `raw_data_manifest_phase1.csv` before it can be used.
- Keep all Phase 1 conclusions as hypotheses until data quality and field validation support them.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PHASE1_ROOT = PROJECT_ROOT.parent
else:
    PHASE1_ROOT = PROJECT_ROOT / "phase1_spinelens_ai"

sys.path.insert(0, str(PHASE1_ROOT / "src"))

from spinelens.gate0b import (  # noqa: E402
    fetch_os_download_candidates,
    os_download_candidate_fieldnames,
    PRIORITY_1_SOURCE_IDS,
    read_csv_rows,
    reachability_fieldnames,
    run_reachability_checks,
    update_acquisition_status_from_reachability,
    utc_now_iso,
    write_csv_rows,
)

DATA = PHASE1_ROOT / "data"
INTERIM = DATA / "interim"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
INTERIM.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = DATA / "source_registry_phase1.csv"
ACQUISITION_PATH = DATA / "source_acquisition_status_phase1.csv"
MANIFEST_PATH = DATA / "raw_data_manifest_phase1.csv"
REACHABILITY_PATH = INTERIM / "gate0b_source_reachability.csv"
OS_DOWNLOAD_CANDIDATES_PATH = INTERIM / "gate0b_os_download_candidates.csv"
READINESS_NOTE_PATH = REPORTS / "gate0b_data_readiness_note.md"

registry_rows = read_csv_rows(REGISTRY_PATH)
acquisition_rows = read_csv_rows(ACQUISITION_PATH)
manifest_rows = read_csv_rows(MANIFEST_PATH)

priority_1 = [row for row in registry_rows if row["source_id"] in PRIORITY_1_SOURCE_IDS]
pd.DataFrame(priority_1)[["source_id", "source_name", "owner", "trust_tier", "url", "license"]]

## Reachability Check

The next cell checks only page/API reachability. It uses `HEAD` first and falls back to a streamed range `GET` only where needed. It does not download bulk data.

In [ ]:
reachability_rows = run_reachability_checks(
    registry_rows,
    source_ids=PRIORITY_1_SOURCE_IDS,
    timeout_seconds=25,
)

write_csv_rows(REACHABILITY_PATH, reachability_rows, reachability_fieldnames())

reachability_df = pd.DataFrame(reachability_rows)
reachability_df[[
    "source_id",
    "reachable",
    "status_code",
    "method",
    "content_type",
    "forensic_status",
    "final_url",
]]

## OS Downloads API Candidate Inventory

This step checks OS open-data download metadata without downloading the files. It records file names, sizes, MD5 hashes supplied by OS, and a cautious download decision.

In [ ]:
os_download_candidates = fetch_os_download_candidates()
write_csv_rows(
    OS_DOWNLOAD_CANDIDATES_PATH,
    os_download_candidates,
    os_download_candidate_fieldnames(),
)

os_candidates_df = pd.DataFrame(os_download_candidates)
os_candidates_df[[
    "source_id",
    "os_product_id",
    "area",
    "file_name",
    "format",
    "size_bytes",
    "md5",
    "download_decision",
]]

## Update Acquisition Ledger

Reachability can raise a source from `not_started` to `source_reachable`, but it must not mark raw data as acquired, audited, cross-checked, or funding-ready.

In [ ]:
updated_acquisition_rows = update_acquisition_status_from_reachability(
    acquisition_rows,
    reachability_rows,
)

write_csv_rows(
    ACQUISITION_PATH,
    updated_acquisition_rows,
    list(acquisition_rows[0].keys()),
)

updated_df = pd.DataFrame(updated_acquisition_rows)
updated_df[updated_df["source_id"].isin(PRIORITY_1_SOURCE_IDS)][[
    "source_id",
    "forensic_status",
    "evidence_level",
    "raw_data_acquired",
    "checksum_recorded",
    "quality_audited",
    "cross_checked",
    "can_support_funding_claim",
    "next_action",
]]

## Manifest Check

The raw-data manifest remains empty until an actual raw file is acquired. This is intentional. It prevents confusing source reachability with usable evidence.

In [ ]:
manifest_df = pd.DataFrame(manifest_rows)
manifest_status = {
    "manifest_path": str(MANIFEST_PATH.relative_to(PHASE1_ROOT)),
    "raw_file_rows": len(manifest_rows),
    "gate0b_interpretation": "No raw files acquired yet; source reachability only.",
}
manifest_status

## Data Readiness Note

The note below is the reviewable Gate 0B output. It should be read before any pedestrian graph or route-scoring notebook is run.

In [ ]:
reachable_count = sum(row["reachable"] == "yes" for row in reachability_rows)
failed = [row for row in reachability_rows if row["reachable"] != "yes"]
safe_small_candidates = [
    row for row in os_download_candidates
    if row["download_decision"] == "safe_small_candidate_for_pavilion_context"
]
checked_at = utc_now_iso()

status_lines = []
for row in reachability_rows:
    status_lines.append(
        f"| {row['source_id']} | {row['reachable']} | {row['status_code']} | {row['forensic_status']} | {row['final_url']} |"
    )

note = "\n".join([
    "# Gate 0B Data Readiness Note",
    "",
    f"Generated: {checked_at}",
    "",
    "## Decision",
    "",
    "Gate 0B has begun. Priority 1 source reachability has been checked, but no raw bulk data has been acquired yet.",
    "",
    f"Reachable Priority 1 sources: {reachable_count} of {len(reachability_rows)}.",
    "",
    "## Source Reachability",
    "",
    "| Source ID | Reachable | HTTP Status | Forensic Status | Final URL |",
    "|---|---:|---:|---|---|",
    *status_lines,
    "",
    "## Raw Data Status",
    "",
    f"Raw manifest rows: {len(manifest_rows)}.",
    "",
    "## OS Download Candidate Inventory",
    "",
    f"Candidate metadata rows: {len(os_download_candidates)}.",
    f"Small safe candidates for immediate review: {len(safe_small_candidates)}.",
    f"Inventory path: {OS_DOWNLOAD_CANDIDATES_PATH.relative_to(PHASE1_ROOT)}.",
    "",
    "No source can support funding-facing claims yet because raw data has not been acquired, checksummed, quality-audited, cross-checked, or field-validated.",
    "",
    "## Next Controlled Step",
    "",
    "Acquire the smallest useful raw extracts for the pedestrian graph and basemap context. Start with OSM pedestrian network extraction inside the Gate 0 boundary, then add authoritative OS/ONS context layers where download size and licence allow.",
    "",
    "## Failed Or Ambiguous Sources",
    "",
    "None." if not failed else "\n".join(f"- {row['source_id']}: {row['notes']}" for row in failed),
])

READINESS_NOTE_PATH.write_text(note, encoding="utf-8")
print(note)